# CDS524 Assignment 1 — Maze Solver with Q-Learning

**Author:** ZHANGZhen
**Date:** March 2026  

## Overview

This notebook implements a **Q-Learning reinforcement learning agent** that learns to navigate a **20×20 complex maze** from a start position `(1,1)` to a goal `(18,18)`.

| Component | Details |
|---|---|
| Game | 20×20 Complex Maze |
| Algorithm | Tabular Q-Learning |
| State Space | All non-wall grid cells (~170 states) |
| Action Space | Up / Down / Left / Right (4 actions) |
| Exploration | ε-Greedy with exponential decay |
| Training | 1200 episodes |

---
## Notebook Structure
1. Install & Imports
2. Maze Definition (State Space & Action Space)
3. Reward Function
4. Environment Class
5. Q-Learning Agent
6. Training Loop
7. Results & Evaluation
8. Discussion

## 1. Install & Imports

In [ ]:
!pip install numpy matplotlib -q

import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import matplotlib.patheffects as pe
import time

np.random.seed(42)
random.seed(42)
print('All imports ready!')

## 2. Maze Definition

### State Space
The **state space** consists of all grid positions `(row, col)` where `MAZE[row][col] == 0` (walkable path).  
A 20×20 grid has 400 cells total, but walls reduce the valid state space to approximately **170 states**.

### Action Space
The agent has **4 discrete actions**: `Up`, `Down`, `Left`, `Right`.  
This gives a Q-table size of approximately **170 × 4 = 680 entries**.

### Maze Complexity
The 20×20 maze features:
- Multiple dead ends requiring backtracking
- Long corridors with no shortcuts
- A winding optimal path of ~39 steps

In [ ]:
# ─────────────────────────────────────────────────────────────
# 20×20 Maze  (0 = walkable path,  1 = wall)
# ─────────────────────────────────────────────────────────────
MAZE = [
    [1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1],
    [1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1],
    [1,0,1,1,1,0,1,0,1,1,1,1,0,1,1,0,1,1,0,1],
    [1,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,1,0,1],
    [1,0,1,0,1,1,1,1,1,0,1,0,1,0,1,1,0,1,0,1],
    [1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,1],
    [1,1,1,0,1,0,1,0,1,1,1,1,1,1,1,1,0,1,0,1],
    [1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1],
    [1,0,1,1,1,0,1,1,1,1,1,0,1,1,0,1,1,1,0,1],
    [1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,1],
    [1,1,1,1,1,1,1,0,1,0,1,0,1,1,1,1,0,1,0,1],
    [1,0,0,0,0,0,1,0,1,0,0,0,1,0,0,0,0,1,0,1],
    [1,0,1,1,0,1,1,0,1,1,1,0,1,0,1,1,1,1,0,1],
    [1,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1],
    [1,0,1,0,1,1,1,1,1,0,1,1,1,0,1,0,1,1,0,1],
    [1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,1],
    [1,1,1,0,1,0,1,0,1,1,1,0,1,1,1,0,1,0,1,1],
    [1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,1,1],
    [1,0,1,1,1,1,1,1,1,0,1,1,1,1,1,0,0,0,0,1],
    [1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1],
]

START = (1, 1)    # Agent start position
GOAL  = (18, 18)  # Agent must reach here
ROWS  = len(MAZE)
COLS  = len(MAZE[0])

# ── Action definitions ──
ACTIONS      = [0, 1, 2, 3]
ACTION_NAMES = ['Up', 'Down', 'Left', 'Right']
ACTION_DELTA = {
    0: (-1,  0),  # Up
    1: ( 1,  0),  # Down
    2: ( 0, -1),  # Left
    3: ( 0,  1),  # Right
}

valid_states = [(r,c) for r in range(ROWS) for c in range(COLS) if MAZE[r][c]==0]
print(f'Maze size    : {ROWS} × {COLS}')
print(f'Start        : {START}')
print(f'Goal         : {GOAL}')
print(f'State space  : {len(valid_states)} valid cells')
print(f'Action space : {len(ACTIONS)} actions → {ACTION_NAMES}')
print(f'Q-table size : {len(valid_states)} × {len(ACTIONS)} = {len(valid_states)*len(ACTIONS)} entries')

In [ ]:
def visualize_maze(path=None, title='Maze Layout', q_table=None):
    fig, ax = plt.subplots(figsize=(9, 9))
    fig.patch.set_facecolor('#0a0e1a')
    ax.set_facecolor('#0a0e1a')

    # Draw cells
    for r in range(ROWS):
        for c in range(COLS):
            if MAZE[r][c] == 1:
                color = '#141e32'
            else:
                color = '#1a2540'
            rect = plt.Rectangle((c, ROWS-1-r), 1, 1,
                                  facecolor=color, edgecolor='#0d1628', lw=0.5)
            ax.add_patch(rect)

    # Overlay Q-value heatmap if provided
    if q_table:
        q_max = max(np.max(v) for v in q_table.values())
        q_min = min(np.max(v) for v in q_table.values())
        for (r,c), qv in q_table.items():
            val = (np.max(qv) - q_min) / max(q_max - q_min, 1)
            color = (0, val*0.7, val)
            rect = plt.Rectangle((c, ROWS-1-r), 1, 1,
                                  facecolor=color, edgecolor='#0d1628', lw=0.3, alpha=0.6)
            ax.add_patch(rect)

    # Draw path
    if path and len(path) > 1:
        px = [p[1]+0.5 for p in path]
        py = [ROWS-0.5-p[0] for p in path]
        ax.plot(px, py, '-', color='#00b4ff', lw=2.5, alpha=0.9,
                path_effects=[pe.Stroke(linewidth=4, foreground='#003366', alpha=0.5), pe.Normal()])
        ax.plot(px, py, 'o', color='#00b4ff', ms=4, alpha=0.7)

    # Start / Goal
    ax.add_patch(plt.Circle((START[1]+0.5, ROWS-0.5-START[0]), 0.35,
                             color='#32c878', zorder=5))
    ax.text(START[1]+0.5, ROWS-0.5-START[0], 'S', ha='center', va='center',
            fontsize=10, fontweight='bold', color='white', zorder=6)

    ax.add_patch(plt.Circle((GOAL[1]+0.5, ROWS-0.5-GOAL[0]), 0.35,
                             color='#ff5050', zorder=5))
    ax.text(GOAL[1]+0.5, ROWS-0.5-GOAL[0], 'G', ha='center', va='center',
            fontsize=10, fontweight='bold', color='white', zorder=6)

    ax.set_xlim(0, COLS); ax.set_ylim(0, ROWS)
    ax.set_aspect('equal')
    ax.axis('off')

    patches = [
        mpatches.Patch(color='#1a2540', label='Path'),
        mpatches.Patch(color='#141e32', label='Wall'),
        mpatches.Patch(color='#32c878', label='Start (S)'),
        mpatches.Patch(color='#ff5050', label='Goal (G)'),
    ]
    if path: patches.append(mpatches.Patch(color='#00b4ff', label='Agent Path'))
    ax.legend(handles=patches, loc='upper right', fontsize=9,
              facecolor='#0a0e1a', edgecolor='#1e2d4a', labelcolor='white')
    ax.set_title(title, fontsize=13, fontweight='bold', color='#00c8ff', pad=12)
    plt.tight_layout()
    plt.savefig('maze_layout.png', dpi=150, facecolor='#0a0e1a', bbox_inches='tight')
    plt.show()

visualize_maze(title='20×20 Maze — Before Training')
print('Maze visualized!')

## 3. Reward Function

The reward function is the core signal that guides Q-learning. It encodes what the agent should optimize:

| Event | Reward | Design Rationale |
|---|---|---|
| Reach goal `(18,18)` | **+200** | Strong positive signal to clearly identify the objective |
| Walk into a wall | **−15** | Penalizes invalid moves; agent learns maze boundaries |
| Revisit a cell | **−3** | Penalizes loops; encourages exploration of new paths |
| Normal step | **−1** | Small cost per step; encourages finding the *shortest* path |

> **Why a step cost?** Without it, the agent reaches the goal but takes a roundabout path. The −1 per step creates pressure to minimize total steps, producing more optimal policies.

In [ ]:
REWARD_GOAL    =  200
REWARD_WALL    =  -15
REWARD_STEP    =   -1
REWARD_REVISIT =   -3

print('Reward function:')
for name, val in [('Goal reached', REWARD_GOAL), ('Wall hit', REWARD_WALL),
                   ('Normal step', REWARD_STEP), ('Revisit cell', REWARD_REVISIT)]:
    bar = '█' * abs(val//5)
    sign = '+' if val > 0 else ''
    print(f'  {name:<15} {sign}{val:>5}  {bar}')

## 4. Environment Class

In [ ]:
class MazeEnv:
    """
    Maze Environment — handles state transitions and reward assignment.

    State  : (row, col) tuple — the agent's current grid position
    Action : integer 0-3 (Up / Down / Left / Right)
    Returns: (next_state, reward, done) on each step
    """

    def reset(self):
        """Reset agent to START at the beginning of each episode."""
        self.pos     = START
        self.visited = {START}
        return self.pos

    def step(self, action):
        """
        Execute one action.
        Returns (next_state, reward, done).
        """
        dr, dc = ACTION_DELTA[action]
        nr, nc = self.pos[0] + dr, self.pos[1] + dc

        # ── Boundary / wall check ──
        if (nr < 0 or nr >= ROWS or nc < 0 or nc >= COLS or MAZE[nr][nc] == 1):
            return self.pos, REWARD_WALL, False   # Stay, penalize

        self.pos = (nr, nc)

        # ── Goal check ──
        if self.pos == GOAL:
            return self.pos, REWARD_GOAL, True    # Episode complete!

        # ── Revisit / normal step ──
        if self.pos in self.visited:
            reward = REWARD_REVISIT
        else:
            reward = REWARD_STEP
            self.visited.add(self.pos)

        return self.pos, reward, False

    def valid_states(self):
        return [(r, c) for r in range(ROWS)
                       for c in range(COLS) if MAZE[r][c] == 0]


# ── Quick sanity test ──
env   = MazeEnv()
state = env.reset()
print(f'Reset → state: {state}')
ns, r, done = env.step(3)   # Move Right
print(f'Move Right → next: {ns}, reward: {r}, done: {done}')
ns, r, done = env.step(0)   # Move Up (into wall)
print(f'Move Up (wall) → next: {ns}, reward: {r}, done: {done}')

## 5. Q-Learning Agent

### The Bellman Update Equation

$$Q(s, a) \leftarrow Q(s, a) + \alpha \Big[ r + \gamma \cdot \max_{a'} Q(s', a') - Q(s, a) \Big]$$

| Symbol | Name | Value | Role |
|---|---|---|---|
| $\alpha$ | Learning rate | 0.15 | How much each update shifts the Q-value |
| $\gamma$ | Discount factor | 0.97 | How much future rewards are valued vs immediate |
| $\varepsilon$ | Exploration rate | 1.0 → 0.02 | Balance random vs greedy action selection |

### ε-Greedy Policy
```
With probability ε   → choose a RANDOM action  (explore unknown paths)
With probability 1-ε → choose the BEST action   (exploit learned knowledge)
```
ε starts at 1.0 (fully random) and decays by ×0.996 each episode, reaching ~0.02 after 1200 episodes.

In [ ]:
ALPHA         = 0.15
GAMMA         = 0.97
EPSILON_START = 1.0
EPSILON_END   = 0.02
EPSILON_DECAY = 0.996
EPISODES      = 1200
MAX_STEPS     = 500


class QLearningAgent:
    """
    Tabular Q-Learning Agent.

    Maintains a Q-table: dict mapping state (r,c) → numpy array of 4 Q-values.
    All Q-values initialised to 0 (optimistic initialisation not used here).
    """

    def __init__(self, env):
        # Q-table: one row per valid state, 4 columns (one per action)
        self.q       = {s: np.zeros(4) for s in env.valid_states()}
        self.epsilon = EPSILON_START

    def choose_action(self, state):
        """
        ε-Greedy action selection.
        Early in training:  mostly explore (ε≈1).
        Late in training :  mostly exploit (ε≈0.02).
        """
        if random.random() < self.epsilon:
            return random.randint(0, 3)             # Explore: random action
        return int(np.argmax(self.q[state]))        # Exploit: best known action

    def update(self, s, a, r, ns, done):
        """
        Bellman update — moves Q(s,a) toward the observed TD target.
        TD target = r + γ * max_a' Q(s', a')   (or just r if terminal)
        """
        td_target = r if done else r + GAMMA * np.max(self.q[ns])
        td_error  = td_target - self.q[s][a]
        self.q[s][a] += ALPHA * td_error            # Incremental update

    def decay_epsilon(self):
        """Reduce exploration rate after each episode."""
        self.epsilon = max(EPSILON_END, self.epsilon * EPSILON_DECAY)

    def greedy_path(self):
        """Extract the greedy (best known) path after training."""
        env   = MazeEnv()
        state = env.reset()
        path  = [state]
        seen  = {state}
        for _ in range(MAX_STEPS):
            action     = int(np.argmax(self.q[state]))
            next_state, _, done = env.step(action)
            if next_state in seen: break     # Loop guard
            path.append(next_state)
            seen.add(next_state)
            state = next_state
            if done: break
        return path


# Epsilon decay curve
eps_curve = [max(EPSILON_END, EPSILON_START * EPSILON_DECAY**e) for e in range(EPISODES)]
fig, ax = plt.subplots(figsize=(8,3))
fig.patch.set_facecolor('#0a0e1a'); ax.set_facecolor('#0d1626')
ax.plot(eps_curve, color='#00b4ff', lw=2)
ax.fill_between(range(EPISODES), eps_curve, alpha=0.15, color='#00b4ff')
ax.set_title('ε-Greedy Decay Schedule', color='#00c8ff', fontsize=12)
ax.set_xlabel('Episode', color='#6482b4')
ax.set_ylabel('Epsilon (ε)', color='#6482b4')
ax.tick_params(colors='#6482b4')
for sp in ax.spines.values(): sp.set_edgecolor('#1e2d4a')
ax.axhline(EPSILON_END, color='#ff5050', lw=1, ls='--', label=f'ε_min = {EPSILON_END}')
ax.legend(facecolor='#0a0e1a', edgecolor='#1e2d4a', labelcolor='white')
plt.tight_layout()
plt.show()
print(f'After {EPISODES} episodes, ε = {eps_curve[-1]:.4f}')

## 6. Training Loop

In [ ]:
def train():
    """
    Main Q-Learning training loop.

    Each episode:
      1. Reset environment to START
      2. Loop: choose action → observe (s', r, done) → update Q-table
      3. Decay epsilon
    """
    env     = MazeEnv()
    agent   = QLearningAgent(env)
    rewards = []
    steps_h = []
    wins    = 0

    for ep in range(EPISODES):
        state     = env.reset()
        total_rew = 0
        steps     = 0

        for _ in range(MAX_STEPS):
            # ── Step 1: Choose action ──
            action = agent.choose_action(state)

            # ── Step 2: Interact with environment ──
            next_state, reward, done = env.step(action)

            # ── Step 3: Update Q-table (Bellman equation) ──
            agent.update(state, action, reward, next_state, done)

            state      = next_state
            total_rew += reward
            steps     += 1

            if done:
                wins += 1
                break

        # ── Step 4: Decay exploration rate ──
        agent.decay_epsilon()

        rewards.append(total_rew)
        steps_h.append(steps)

        if (ep + 1) % 200 == 0:
            print(f'  Episode {ep+1:4d}/{EPISODES}  |  '
                  f'ε={agent.epsilon:.3f}  |  '
                  f'AvgReward={np.mean(rewards[-100:]):8.1f}  |  '
                  f'AvgSteps={np.mean(steps_h[-100:]):6.1f}  |  '
                  f'WinRate={wins/(ep+1)*100:.1f}%')

    print(f'\nTraining complete!')
    print(f'Goal reached: {wins}/{EPISODES} episodes ({wins/EPISODES*100:.1f}%)')
    return agent, rewards, steps_h, wins


print('Starting training...\n')
t0 = time.time()
agent, rewards, steps_h, wins = train()
print(f'Time elapsed: {time.time()-t0:.1f}s')

## 7. Results & Evaluation

In [ ]:
# ── Training curves ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#080c16')
fig.suptitle('Q-Learning Training Performance — 20×20 Maze', color='#00c8ff', fontsize=13)

w = 40
sr = np.convolve(rewards,  np.ones(w)/w, mode='valid')
ss = np.convolve(steps_h,  np.ones(w)/w, mode='valid')
xs = range(w-1, len(rewards))

for ax in axes:
    ax.set_facecolor('#0c1222')
    ax.tick_params(colors='#6482b4')
    for sp in ax.spines.values(): sp.set_edgecolor('#1e2d4a')

axes[0].plot(rewards, alpha=0.15, color='#00b4ff')
axes[0].plot(xs, sr, color='#00b4ff', lw=2.5, label='Moving avg (40 ep)')
axes[0].fill_between(xs, sr, alpha=0.1, color='#00b4ff')
axes[0].set_title('Episode Reward', color='#00c8ff')
axes[0].set_xlabel('Episode', color='#6482b4')
axes[0].set_ylabel('Total Reward', color='#6482b4')
axes[0].legend(facecolor='#0a0e1a', edgecolor='#1e2d4a', labelcolor='white')

axes[1].plot(steps_h, alpha=0.15, color='#50dc82')
axes[1].plot(xs, ss, color='#50dc82', lw=2.5, label='Moving avg (40 ep)')
axes[1].fill_between(xs, ss, alpha=0.1, color='#50dc82')
axes[1].set_title('Steps per Episode', color='#00c8ff')
axes[1].set_xlabel('Episode', color='#6482b4')
axes[1].set_ylabel('Steps', color='#6482b4')
axes[1].legend(facecolor='#0a0e1a', edgecolor='#1e2d4a', labelcolor='white')

plt.tight_layout()
plt.savefig('training_results.png', dpi=150, facecolor='#080c16', bbox_inches='tight')
plt.show()
print('Chart saved: training_results.png')

In [ ]:
# ── Learned path ──
best_path = agent.greedy_path()
print('=== Learned Policy Evaluation ===')
print(f'Path length  : {len(best_path)} steps')
print(f'Goal reached : {best_path[-1] == GOAL}')
print(f'Path         : {best_path}')

visualize_maze(path=best_path, title=f'Learned Optimal Path ({len(best_path)} steps)')

In [ ]:
# ── Q-Table heatmap ──
fig, ax = plt.subplots(figsize=(9, 9))
fig.patch.set_facecolor('#0a0e1a')
ax.set_facecolor('#0a0e1a')

for r in range(ROWS):
    for c in range(COLS):
        color = '#101828' if MAZE[r][c] == 1 else '#1a2540'
        rect = plt.Rectangle((c, ROWS-1-r), 1, 1, facecolor=color,
                              edgecolor='#0d1628', lw=0.4)
        ax.add_patch(rect)

q_max = max(np.max(v) for v in agent.q.values())
q_min = min(np.max(v) for v in agent.q.values())
for (r,c), qv in agent.q.items():
    t = (np.max(qv) - q_min) / max(q_max - q_min, 1)
    color = (0, t*0.8, t)
    rect = plt.Rectangle((c+0.05, ROWS-0.95-r), 0.9, 0.9,
                          facecolor=color, edgecolor='none', alpha=0.85)
    ax.add_patch(rect)

# Overlay path
if best_path:
    px = [p[1]+0.5 for p in best_path]
    py = [ROWS-0.5-p[0] for p in best_path]
    ax.plot(px, py, '--', color='#ffffff', lw=1.5, alpha=0.6)

ax.add_patch(plt.Circle((START[1]+0.5, ROWS-0.5-START[0]), 0.35, color='#32c878', zorder=5))
ax.text(START[1]+0.5, ROWS-0.5-START[0], 'S', ha='center', va='center',
        fontsize=9, fontweight='bold', color='white', zorder=6)
ax.add_patch(plt.Circle((GOAL[1]+0.5, ROWS-0.5-GOAL[0]), 0.35, color='#ff5050', zorder=5))
ax.text(GOAL[1]+0.5, ROWS-0.5-GOAL[0], 'G', ha='center', va='center',
        fontsize=9, fontweight='bold', color='white', zorder=6)

from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
sm = ScalarMappable(cmap='cool', norm=Normalize(vmin=q_min, vmax=q_max))
plt.colorbar(sm, ax=ax, label='Max Q-Value', shrink=0.8)
ax.set_xlim(0, COLS); ax.set_ylim(0, ROWS); ax.set_aspect('equal'); ax.axis('off')
ax.set_title('Q-Table Heatmap\n(Brighter = Higher Value = Closer to Goal)',
             color='#00c8ff', fontsize=12, pad=10)
plt.tight_layout()
plt.savefig('q_table_heatmap.png', dpi=150, facecolor='#0a0e1a', bbox_inches='tight')
plt.show()
print('Heatmap saved: q_table_heatmap.png')

In [ ]:
# ── Summary ──
print('=' * 50)
print('       TRAINING SUMMARY')
print('=' * 50)
print(f'  Maze size          : {ROWS} × {COLS}')
print(f'  State space        : {len(agent.q)} cells')
print(f'  Action space       : {len(ACTIONS)} directions')
print(f'  Episodes           : {EPISODES}')
print(f'  Goal reached       : {wins}/{EPISODES} ({wins/EPISODES*100:.1f}%)')
print(f'  Final epsilon      : {agent.epsilon:.4f}')
print(f'  Last 100ep reward  : {np.mean(rewards[-100:]):.1f}')
print(f'  Last 100ep steps   : {np.mean(steps_h[-100:]):.1f}')
print(f'  Best path length   : {len(best_path)} steps')
print(f'  Goal reached (test): {best_path[-1] == GOAL}')
print('=' * 50)

## 8. Discussion

### Learning Behaviour

The reward and steps curves reveal three distinct training phases:

**Phase 1 (Episodes 1–300) — Random Exploration:** ε is high, so the agent takes mostly random actions. Rewards are very negative and steps are near the MAX_STEPS cap. The Q-table is slowly being populated from random encounters with the goal.

**Phase 2 (Episodes 300–800) — Transition:** ε decays into the mid-range. The agent starts exploiting partial knowledge. Rewards improve noticeably and steps begin to decrease. Dead ends are increasingly avoided.

**Phase 3 (Episodes 800–1200) — Convergence:** ε approaches 0.02. The agent reliably navigates to the goal. Reward and steps curves plateau, indicating a stable policy has been learned.

### Challenges & Solutions

| Challenge | Solution |
|---|---|
| Agent loops in dead ends | Revisit penalty (−3) discourages returning to known cells |
| Slow goal discovery | High initial ε (1.0) ensures thorough random exploration |
| Long training needed | Increased episodes to 1200 for the larger 20×20 maze |
| Agent takes long paths | Step cost (−1) creates incentive to minimize path length |
| Unstable early learning | Higher learning rate (α=0.15) accelerates initial Q-value updates |

### Hyperparameter Analysis

- **α = 0.15**: Higher than default (0.1) because the larger maze needs faster convergence. Too high (>0.5) would cause oscillation.
- **γ = 0.97**: High discount factor values future rewards strongly — important for long-horizon maze navigation where the goal is ~40 steps away.
- **ε decay = 0.996**: Calibrated so ε reaches 0.02 around episode 1000, leaving 200 episodes of near-pure exploitation for policy refinement.